# MS4215 · Python Training 02
## Lab 1 — Spotify Track Energy

**Original lab:** MS4215 Lab 1, Sinéad Moylett (R).  
**This notebook:** a Python teaching adaptation, following the original six question groups **in their original order**.

**Core question:** Which measurable musical features are associated with a track's energy?

**Approach:** Inspect → Visualise → Correlate → Simple regression → Multiple regression → Diagnose → Explain.

The course file contains **114,000 Spotify tracks**. Following the original lab, we analyse a **reproducible random sample of 3,000 tracks**. Python's random sample uses a different random-number generator from R, so the exact selected records and estimated coefficients need not match the supplied R solutions.

**Use:** Run one cell at a time with Shift+Enter. Before each model or chart, write down what you think it will show.


## Set up the real course dataset

The original Spotify data file is available on Erik's website as a ZIP. The code checks for a local CSV first (as in the downloadable course pack), then obtains the published ZIP automatically in Colab.

**Variables to notice:** `energy` (outcome, 0–1), `loudness` (dB), `acousticness`, `valence`, `danceability`, `tempo`, `popularity` and `track_genre`.


In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pandas.plotting import scatter_matrix

file_name = "lab1_dataset_spotify.csv"
candidates = [Path("data") / file_name, Path("../data") / file_name, Path(file_name)]
existing = next((p for p in candidates if p.is_file()), None)
if existing:
    spotify_full = pd.read_csv(existing)
    print("Loaded local course file:", existing)
else:
    source = ("https://raw.githubusercontent.com/ErikDownes/"
              "ErikDownes.github.io/main/resources/ms4215/archives/"
              "lab1_dataset_spotify.zip")
    with ZipFile(BytesIO(urlopen(source).read())) as archive:
        names = [n for n in archive.namelist() if n.lower().endswith(".csv")]
        if not names:
            raise FileNotFoundError("The published Spotify ZIP has no CSV file.")
        with archive.open(names[0]) as csv_file:
            spotify_full = pd.read_csv(csv_file)
    print("Loaded published course ZIP:", names[0])

spotify = spotify_full.sample(n=3000, random_state=42).copy()
print("Full data:", spotify_full.shape, "Sample:", spotify.shape)
display(spotify.head())


## 1 · Explore the data (original lab question i)

**Statistical questions:**
- What is one observation?
- Which fields are categorical and which are numerical?
- Are there missing values?
- What is the minimum and maximum energy in our 3,000-track sample?

**Concept:** A value close to 1 represents high energy, not necessarily musical quality or popularity.


In [ ]:
print(spotify.info())
print("\nMissing values in sample (top 10):")
print(spotify.isna().sum().sort_values(ascending=False).head(10))
print("\nEnergy range:", (spotify["energy"].min(), spotify["energy"].max()))
display(spotify[["energy", "loudness", "acousticness",
                 "valence", "danceability", "tempo"]].describe().T.round(3))


**Pause and explain:** What does it mean that `energy` is bounded between 0 and 1? Why might this matter when we later use *linear* regression?


## 2 · Histograms and summary statistics (original lab question ii)

Plot three distributions: `energy`, `loudness` and `acousticness`.

**Choose** mean + SD for roughly symmetric data; median + IQR for heavily skewed data or prominent outliers. There is no substitute for inspecting the distribution.


In [ ]:
fields = ["energy", "loudness", "acousticness"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, field in zip(axes, fields):
    ax.hist(spotify[field].dropna(), bins=30)
    ax.set(title=field, xlabel=field, ylabel="Tracks")
plt.tight_layout()
plt.show()

stats = spotify[fields].agg(["mean", "std", "median", "min", "max"]).T
stats["IQR"] = spotify[fields].quantile(.75) - spotify[fields].quantile(.25)
display(stats.round(3))


**Your decision:** For each variable, select mean/SD or median/IQR. Justify the choice from the **shape**, not because one statistic is easier to calculate.


## 3 · Correlation and scatterplot matrix (original lab question iii)

Pearson's correlation coefficient measures **linear association** from −1 to +1.

A correlation close to zero does not rule out a strong **nonlinear** relationship. Correlation is not proof of causation.

**Prediction:** Does loudness, acousticness, tempo, valence or danceability have the strongest linear relationship with energy?


In [ ]:
variables = ["energy", "loudness", "acousticness",
             "valence", "danceability", "tempo"]
correlations = spotify[variables].corr(method="pearson")
display(correlations.round(3))
association = (correlations["energy"].drop("energy")
               .sort_values(key=lambda s: s.abs(), ascending=False))
print("\nPredictors ranked by absolute correlation with energy:")
display(association.round(3).to_frame("Pearson r"))

axes = scatter_matrix(spotify[variables], figsize=(12, 12),
                      diagonal="hist", alpha=.12)
plt.suptitle("Spotify variables: scatterplot matrix", y=1.01)
plt.tight_layout()
plt.show()


**Explain:** Which predictor came first? Is its correlation positive or negative? Describe the direction and strength *in words*. Avoid interpreting correlation as a causal effect.


## 4 · Simple linear regression (original lab question iv)

We now estimate

\[\widehat{\mathrm{energy}} = \hat\beta_0 + \hat\beta_1 x\]

where `x` is the predictor with the strongest absolute Pearson correlation with energy.

- **Intercept:** the model's predicted energy when x = 0; it may be outside the observed or meaningful range.
- **Slope:** expected change in predicted energy for a one-unit increase in x.
- **R²:** fraction of variation in energy explained *in this sample* by the fitted model.


In [ ]:
best_predictor = association.index[0]
simple_frame = spotify[["energy", best_predictor]].dropna()
X_simple = sm.add_constant(simple_frame[[best_predictor]])
simple_model = sm.OLS(simple_frame["energy"], X_simple).fit()
print("Chosen predictor:", best_predictor)
print(simple_model.summary())
print(f"\nFitted equation: energy_hat = {simple_model.params['const']:.4f} "
      f"+ ({simple_model.params[best_predictor]:.4f}) × {best_predictor}")
print(f"R-squared = {simple_model.rsquared:.3f} "
      f"({simple_model.rsquared*100:.1f}% of sample variation)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(simple_frame[best_predictor], simple_frame["energy"], alpha=.12, s=9)
ordered = simple_frame[best_predictor].sort_values()
ax.plot(ordered, simple_model.predict(sm.add_constant(
    ordered.to_frame(name=best_predictor), has_constant="add")))
ax.set(xlabel=best_predictor, ylabel="energy", title="Simple linear regression")
plt.tight_layout()
plt.show()


**Interpretation checkpoint:** Write the fitted equation with the actual estimates, interpret the slope **with units**, and state the R² as a percentage. Does a high R² prove a causal relationship? No.


## 5 · Multiple linear regression (original lab question v)

This model uses **four predictors simultaneously**: loudness, acousticness, valence and danceability.

A **partial regression coefficient** describes the change in predicted energy associated with its predictor **holding the other included predictors constant**. It is not the same as a simple correlation.

Adjusted R² penalises unnecessary complexity. A p-value is evidence against a zero coefficient under the model assumptions; it is not a measure of practical importance.


In [ ]:
features = ["loudness", "acousticness", "valence", "danceability"]
model_frame = spotify[["energy", *features]].dropna()
X_multi = sm.add_constant(model_frame[features])
multiple_model = sm.OLS(model_frame["energy"], X_multi).fit()
print(multiple_model.summary())
result_table = pd.DataFrame({
    "coefficient": multiple_model.params,
    "p_value": multiple_model.pvalues,
    "significant_5pct": multiple_model.pvalues < .05
})
display(result_table.round(4))
print(f"Simple adjusted R²:   {simple_model.rsquared_adj:.3f}")
print(f"Multiple adjusted R²: {multiple_model.rsquared_adj:.3f}")


**Pause and explain:**
1. What does the coefficient for `loudness` mean if the other three predictors stay fixed?
2. Why might a predictor's coefficient change between simple and multiple regression?
3. How does adjusted R² change?
4. Which predictors meet the 5% significance threshold?
5. Does statistical significance guarantee the model is useful in practice?


## 6 · Residual diagnostics (original lab question vi)

A residual is observed energy − fitted energy. Regression is not complete until we ask whether the model is credible.

Look for:
- **Nonlinearity:** an obvious curve in residuals vs fitted values.
- **Unequal variance:** a funnel or other systematic change in spread.
- **Non-normality:** strong departures from the reference line in a Q–Q plot.
- **Dependence/outliers:** a model can fail even when its regression summary looks impressive.

Energy is bounded in [0,1], so linear predictions and constant-variance assumptions deserve particular care.


In [ ]:
residuals = multiple_model.resid
fitted = multiple_model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(fitted, residuals, s=9, alpha=.25)
axes[0].axhline(0, linestyle="--")
axes[0].set(title="Residuals vs fitted", xlabel="Fitted energy", ylabel="Residual")

axes[1].hist(residuals, bins=35)
axes[1].set(title="Residual distribution", xlabel="Residual", ylabel="Tracks")

sm.qqplot(residuals, line="45", ax=axes[2])
axes[2].set(title="Normal Q–Q plot")
plt.tight_layout()
plt.show()

print("Number of fitted energies outside [0,1]:",
      int(((fitted < 0) | (fitted > 1)).sum()))


## 7 · Communicate the result: analysis is an argument, not a printout

Complete the statements with your **actual outputs**:

**Question** — I investigated whether musical audio features are associated with track energy.

**Evidence** — The strongest simple linear association was with ____ (r = ____). A simple model explained ____% of sample energy variation.

**Model** — With four predictors, the adjusted R² was ____. The coefficient for ____ suggests ____ when other included predictors are held constant.

**Limitations** — These are observational features, so association does not demonstrate causation. The residual plots suggest ____; predictions outside 0–1 would also be questionable.

**Conclusion** — The results support ____, but they do not establish ____.

This is the written interpretation that should accompany a portfolio graph.


## 8 · Independent challenges (scaffold fades)

**A.** Does popularity have much linear correlation with energy? Calculate it and interpret it.

**B.** Fit `energy ~ tempo` alone. Compare its R² with the strongest single-predictor model.

**C.** Replace the sample with a new reproducible random sample (`random_state=7`). Do the results change slightly? Why?

**D.** Make a bar chart of median energy by track genre for the ten most common genres in the sample. Why might median be preferable for some comparisons?

**E.** Propose a research question that *cannot* be resolved by this dataset. Explain what additional evidence would be necessary.

### Recall cues
`distribution` | `correlation` | `slope` | `intercept` | `R²` | `partial slope` | `adjusted R²` | `residuals` | `causation`

**Next university lab:** Espresso Extraction — design matrix, normal equation, tests and multicollinearity.
